<a href="https://colab.research.google.com/github/ahmedaeldahmy/AAI-521-Group4-Eurosat/blob/akua%2Fmodelb/notebooks/02_Akua_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/ahmedaeldahmy/AAI-521-Group4-Eurosat.git /content/AAI-521-Group4-Eurosat

Cloning into '/content/AAI-521-Group4-Eurosat'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 70 (delta 18), reused 7 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 783.38 KiB | 4.28 MiB/s, done.
Resolving deltas: 100% (18/18), done.


# 1. Download EuroSAT_RGB.zip

reads data/split.csv and data/class_map.json

In [4]:
import hashlib
import json
import zipfile
import subprocess
from pathlib import Path

import pandas as pd
import tensorflow as tf

SEED = 42
tf.keras.utils.set_random_seed(SEED)

base = Path("/content") if Path("/content").exists() else Path.cwd()
candidates = [
    base / "AAI-521-Group4-Eurosat",
    Path.cwd(),
    Path.cwd().parent,
]
REPO = next(p for p in candidates if (p / "data" / "split.csv").exists())
RAW_DIR = REPO / "raw_dataset"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://zenodo.org/api/records/7711810/files/EuroSAT_RGB.zip/content"
ZIP_PATH = RAW_DIR / "EuroSAT_RGB.zip"
EXPECTED_MD5 = "f46e308c4d50d4bf32fedad2d3d62f3b"

if not ZIP_PATH.exists():
    print("Downloading from Zenodo via wget...")
    temporary_zip = RAW_DIR / "EuroSAT_RGB.download"
    # Fallback to system wget if requests gets a 403
    try:
        subprocess.run([
            "wget", "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "-O", str(temporary_zip), DATA_URL
        ], check=True)
    except Exception as e:
        import requests
        print("Wget failed, trying requests...")
        with requests.get(
            DATA_URL, stream=True, timeout=(30, 180),
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"},
        ) as response:
            response.raise_for_status()
            with temporary_zip.open("wb") as out:
                for block in response.iter_content(chunk_size=1024 * 1024):
                    if block:
                        out.write(block)
    temporary_zip.replace(ZIP_PATH)

file_md5 = hashlib.md5()
with ZIP_PATH.open("rb") as zipped:
    for block in iter(lambda: zipped.read(1024 * 1024), b""):
        file_md5.update(block)
assert file_md5.hexdigest() == EXPECTED_MD5

marker = RAW_DIR / "extraction_complete.txt"
if not marker.exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_DIR)
    marker.write_text("EuroSAT RGB extracted\n")

split_df = pd.read_csv(REPO / "data" / "split.csv")
class_map = json.loads((REPO / "data" / "class_map.json").read_text())
CLASS_NAMES = class_map["class_names"]
print(REPO)
print(split_df["split"].value_counts().to_dict())

/content/AAI-521-Group4-Eurosat
{'train': 18900, 'val': 4050, 'test': 4050}


# 2. Build train, val, and test sets that resize 64×64 to 224×224 and call mobilenet_v2.preprocess_input.

EuroSAT image is 64 x 64 pixels. MobileNetV2 was built for 224 by 224, so the code stretched every image up to that size before preprocess_input scales the colors into the range the pretrained model expects

In [5]:
def load_split(split, batch_size=32):
    part = split_df.loc[split_df["split"] == split]
    paths = [str(RAW_DIR / name) for name in part["filename"]]
    labels = part["label"].astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def decode(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.ensure_shape(image, (64, 64, 3))
        image = tf.image.resize(image, (224, 224))
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        return image, label

    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(10000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = load_split("train")
val_ds = load_split("val")
test_ds = load_split("test")

# 3. Build model (Freeze base - do not learn during training run)

This section loads MobileNetV2 with knowledge from ImageNet, drops ImageNet’s 1,000-class head, and freezes those layers so they do not change yet. Then it shrink the features, drop 20% of them to reduce overfitting, and output 10 land cover probabilities.

compile sets the learning rate, the loss, and accuracy. summary() prints the layer list.


In [6]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation="softmax")(x)
model_mobile_net = models.Model(inputs, outputs, name="mobilenetv2")
model_mobile_net.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_mobile_net.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# 4. Train the model

training with early stopping = 3

In [ ]:
head_history = model_mobile_net.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
    ],
)

Epoch 1/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 1250s 2s/step - accuracy: 0.8655 - loss: 0.4099 - val_accuracy: 0.9086 - val_loss: 0.2688
Epoch 2/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 1294s 2s/step - accuracy: 0.9235 - loss: 0.2210 - val_accuracy: 0.9227 - val_loss: 0.2289
Epoch 3/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 1269s 2s/step - accuracy: 0.9353 - loss: 0.1912 - val_accuracy: 0.9289 - val_loss: 0.2043
Epoch 4/10
 69/591 ━━━━━━━━━━━━━━━━━━━━ 14:02 2s/step - accuracy: 0.9525 - loss: 0.1439

# 5. unfreeze and fine tune model

early stopping on validation loss.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model_mobile_net.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
fine_history = model_mobile_net.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
    ],
)